In [30]:
import pandas as pd
import numpy as np

def build_model_dataset(df_raw, step_minutes=5, horizon_minutes=30):
    """
    Pour chaque alerte, génère une observation toutes les `step_minutes` minutes.
    
    Target : y=1 si aucun éclair CG dans les `horizon_minutes` qui suivent
             (= l'alerte est "finie" à cet horizon)
    """
    records = []
    
    # On travaille uniquement sur les éclairs dans la zone alerte (< 20 km)
    df_alert = df_raw[df_raw["airport_alert_id"].notna()].copy()
    df_alert["date"] = pd.to_datetime(df_alert["date"], utc=True)
    
    # Zone contexte (20-30 km) — séparée
    df_context = df_raw[
        (df_raw["dist"] > 20) & (df_raw["dist"] <= 30)
    ].copy()
    df_context["date"] = pd.to_datetime(df_context["date"], utc=True)
    
    for (airport, alert_id), grp in df_alert.groupby(["airport", "airport_alert_id"]):
        grp = grp.sort_values("date")
        grp_cg = grp[~grp["icloud"]]  # uniquement les CG dans la zone alerte
        
        if len(grp_cg) == 0:
            continue
        
        alert_start = grp["date"].min()
        alert_end   = grp_cg["date"].max()  # dernier CG = fin de l'alerte
        
        # Observations régulières pendant toute la durée de l'alerte
        obs_times = pd.date_range(
            start=alert_start + pd.Timedelta(minutes=step_minutes),
            end=alert_end,
            freq=f"{step_minutes}min"
        )
        
        # Éclairs de contexte pour cet aéroport
        grp_ctx = df_context[df_context["airport"] == airport]
        
        for t in obs_times:
            # TARGET : plus aucun CG dans les `horizon_minutes` suivants ?
            future_cg = grp_cg[
                (grp_cg["date"] > t) & 
                (grp_cg["date"] <= t + pd.Timedelta(minutes=horizon_minutes))
            ]
            y = 1 if len(future_cg) == 0 else 0
            
            features = compute_features_at_time(
                df_alert=grp,
                df_context=grp_ctx,
                t=t
            )
            
            records.append({
                "airport": airport,
                "alert_id": alert_id,
                "obs_time": t,
                "t_elapsed_min": (t - alert_start).total_seconds() / 60,
                "y": y,
                **features
            })
    
    return pd.DataFrame(records)

In [54]:
import pandas as pd
import numpy as np


# ─────────────────────────────────────────────────────────────────────────────
# CHARGEMENT ET PRÉPARATION
# ─────────────────────────────────────────────────────────────────────────────

def load_and_prepare(csv_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Charge le CSV brut, parse les dates, filtre Pise 2016 IC,
    et sépare zone alerte (< 20 km) et zone contexte (20-30 km).

    Retourne : (df_alert, df_context)
    """
    df = pd.read_csv(csv_path)

    # Parsing de la date
    df["date"] = pd.to_datetime(df["date"], utc=True)

    # Zone alerte : éclairs avec une alerte assignée (dist < 20 km)
    df_alert = df[df["airport_alert_id"].notna()].copy()

    # Zone contexte : éclairs entre 20 et 30 km (pas d'alerte assignée)
    df_context = df[
        (df["dist"] > 20) & (df["dist"] <= 30)
    ].copy()

    return df_alert, df_context


# ─────────────────────────────────────────────────────────────────────────────
# FONCTION CENTRALE : FEATURES À UN INSTANT T
# ─────────────────────────────────────────────────────────────────────────────

def compute_features_at_time(
    grp_alert: pd.DataFrame,
    grp_context: pd.DataFrame,
    t: pd.Timestamp,
    windows: list[int] = [5, 15, 30]
) -> dict:
    """
    Calcule toutes les features pour une alerte donnée à l'instant t.

    Paramètres
    ----------
    grp_alert   : éclairs de l'alerte (dist < 20 km), toutes dates confondues
    grp_context : éclairs de la zone contexte (20-30 km) pour le même aéroport
    t           : instant d'observation (pd.Timestamp, UTC)
    windows     : fenêtres glissantes en minutes [5, 15, 30]

    Retourne
    --------
    dict de features — uniquement basé sur les données AVANT t (pas de leakage)
    """
    features = {}

    # Données passées uniquement (strict < t pour éviter tout leakage)
    past_alert   = grp_alert[grp_alert["date"] < t]
    past_context = grp_context[grp_context["date"] < t]

    past_cg = past_alert[~past_alert["icloud"]]   # CG zone alerte
    past_ic = past_alert[past_alert["icloud"]]    # IC zone alerte
    past_cg_ctx = past_context[~past_context["icloud"]]  # CG zone contexte

    # ── Temps écoulé depuis le début de l'alerte ──────────────────────────────
    if len(past_alert) > 0:
        alert_start = past_alert["date"].min()
        features["t_elapsed_min"] = (t - alert_start).total_seconds() / 60
    else:
        features["t_elapsed_min"] = 0.0

    # ── Temps depuis le dernier CG ────────────────────────────────────────────
    if len(past_cg) > 0:
        last_cg_time = past_cg["date"].max()
        features["minutes_since_last_cg"] = (t - last_cg_time).total_seconds() / 60
    else:
        features["minutes_since_last_cg"] = features["t_elapsed_min"]

    # ── Temps depuis le dernier IC ────────────────────────────────────────────
    if len(past_ic) > 0:
        last_ic_time = past_ic["date"].max()
        features["minutes_since_last_ic"] = (t - last_ic_time).total_seconds() / 60
    else:
        features["minutes_since_last_ic"] = features["t_elapsed_min"]

    # ── Features par fenêtre glissante ────────────────────────────────────────
    for w in windows:
        cutoff = t - pd.Timedelta(minutes=w)
        window_cg  = past_cg[past_cg["date"] >= cutoff]
        window_ic  = past_ic[past_ic["date"] >= cutoff]
        window_all = past_alert[past_alert["date"] >= cutoff]
        window_cg_ctx = past_cg_ctx[past_cg_ctx["date"] >= cutoff]

        # Comptages
        features[f"cg_count_{w}min"]      = len(window_cg)
        features[f"ic_count_{w}min"]      = len(window_ic)
        features[f"total_count_{w}min"]   = len(window_all)
        features[f"cg_ctx_count_{w}min"]  = len(window_cg_ctx)

        # Amplitude CG
        if len(window_cg) > 0:
            features[f"mean_abs_amp_cg_{w}min"] = window_cg["amplitude"].abs().mean()
            features[f"std_abs_amp_cg_{w}min"]  = window_cg["amplitude"].abs().std(ddof=0)
        else:
            features[f"mean_abs_amp_cg_{w}min"] = 0.0
            features[f"std_abs_amp_cg_{w}min"]  = 0.0

        # Distance CG dans la zone alerte
        if len(window_cg) > 0:
            features[f"mean_dist_cg_{w}min"] = window_cg["dist"].mean()
            features[f"max_dist_cg_{w}min"]  = window_cg["dist"].max()
            features[f"std_dist_cg_{w}min"]  = window_cg["dist"].std(ddof=0)
        else:
            features[f"mean_dist_cg_{w}min"] = np.nan
            features[f"max_dist_cg_{w}min"]  = np.nan
            features[f"std_dist_cg_{w}min"]  = np.nan

        # Ratio contexte / alerte (signal d'alimentation depuis l'extérieur)
        total_in_window = len(window_cg) + len(window_cg_ctx)
        if total_in_window > 0:
            features[f"ratio_ctx_alert_{w}min"] = len(window_cg_ctx) / total_in_window
        else:
            features[f"ratio_ctx_alert_{w}min"] = np.nan

    # ── Tendances (comparaison fenêtre courte vs fenêtre longue) ──────────────
    # Un ratio qui tend vers 0 signale un orage qui faiblit
    if features["cg_count_30min"] > 0:
        features["cg_trend_5_30"] = features["cg_count_5min"] / features["cg_count_30min"]
    else:
        features["cg_trend_5_30"] = np.nan

    if features["cg_count_30min"] > 0:
        features["cg_trend_15_30"] = features["cg_count_15min"] / features["cg_count_30min"]
    else:
        features["cg_trend_15_30"] = np.nan

    # Tendance de distance (l'orage s'éloigne-t-il ?)
    dist_5  = features.get("mean_dist_cg_5min")
    dist_30 = features.get("mean_dist_cg_30min")
    if dist_5 is not None and dist_30 is not None and not np.isnan(dist_5) and not np.isnan(dist_30):
        features["dist_trend_5_30"] = dist_5 - dist_30  # positif = s'éloigne
    else:
        features["dist_trend_5_30"] = np.nan

    # ── Cumuls depuis le début de l'alerte ────────────────────────────────────
    features["cum_cg_count"]  = len(past_cg)
    features["cum_ic_count"]  = len(past_ic)
    features["cum_all_count"] = len(past_alert)

    # ── Features temporelles cycliques ────────────────────────────────────────
    features["sin_hour"]  = np.sin(2 * np.pi * t.hour / 24)
    features["cos_hour"]  = np.cos(2 * np.pi * t.hour / 24)
    features["sin_month"] = np.sin(2 * np.pi * t.month / 12)
    features["cos_month"] = np.cos(2 * np.pi * t.month / 12)

    return features


# ─────────────────────────────────────────────────────────────────────────────
# CONSTRUCTION DU DATASET COMPLET
# ─────────────────────────────────────────────────────────────────────────────
 
def build_model_dataset(
    df_alert: pd.DataFrame,
    df_context: pd.DataFrame,
    step_minutes: int = 5,
    horizon_minutes: int = 30
) -> pd.DataFrame:
    """
    Génère le dataset de modélisation.
 
    Pour chaque alerte, on observe toutes les `step_minutes` minutes.
    À chaque instant t :
      - y = 1 si aucun CG dans les `horizon_minutes` suivants (fin imminente)
      - y = 0 sinon
 
    Paramètres
    ----------
    df_alert        : zone alerte (dist < 20 km), issu de load_and_prepare()
    df_context      : zone contexte (20-30 km), issu de load_and_prepare()
    step_minutes    : pas d'échantillonnage temporel (défaut : 5 min)
    horizon_minutes : horizon de prédiction pour la target (défaut : 30 min)
    """
    records = []
    groups = list(df_alert.groupby(["airport", "airport_alert_id"]))
    total = len(groups)
 
    for i, ((airport, alert_id), grp) in enumerate(groups):
        if i % 100 == 0:
            print(f"  Traitement alerte {i+1}/{total}...", end="\r")
 
        grp = grp.sort_values("date")
        grp_cg = grp[~grp["icloud"]]
 
        # On ignore les alertes sans aucun CG (pas de target possible)
        if len(grp_cg) == 0:
            continue
 
        alert_start  = grp["date"].min()
        last_cg_time = grp_cg["date"].max()  # fin réelle de l'alerte
 
        # Contexte : même aéroport seulement
        grp_ctx = df_context[df_context["airport"] == airport].copy()
 
        # Fenêtres d'observation : de step_minutes après le début jusqu'à la fin
        obs_times = pd.date_range(
            start=alert_start + pd.Timedelta(minutes=step_minutes),
            end=last_cg_time,
            freq=f"{step_minutes}min"
        )
 
        for t in obs_times:
            # ── Target ────────────────────────────────────────────────────────
            future_cg = grp_cg[
                (grp_cg["date"] > t) &
                (grp_cg["date"] <= t + pd.Timedelta(minutes=horizon_minutes))
            ]
            y = 1 if len(future_cg) == 0 else 0
 
            # ── Features ──────────────────────────────────────────────────────
            feats = compute_features_at_time(
                grp_alert=grp,
                grp_context=grp_ctx,
                t=t
            )
 
            records.append({
                "airport":           airport,
                "alert_id":          alert_id,
                "obs_time":          t,
                "year":              t.year,
                "last_cg_time":      last_cg_time,
                "remaining_minutes": (last_cg_time - t).total_seconds() / 60,
                "y":                 y,
                **feats
            })
 
    print(f"\nDataset construit : {len(records)} observations.")
    return pd.DataFrame(records)


# ─────────────────────────────────────────────────────────────────────────────
# SPLIT TEMPOREL
# ─────────────────────────────────────────────────────────────────────────────



In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import lightgbm as lgb

df_model = build_model_dataset(df_raw)

# Split temporel : 2016-2023 train, 2024-2025 test
df_train = df_model[df_model["obs_time"].dt.year <= 2023]
df_test  = df_model[df_model["obs_time"].dt.year >= 2024]

feature_cols = [c for c in df_model.columns 
                if c not in ["airport", "alert_id", "obs_time", "y"]]

X_train, y_train = df_train[feature_cols], df_train["y"]
X_test,  y_test  = df_test[feature_cols],  df_test["y"]

# Calcul du poids positif pour l'imbalance
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = lgb.LGBMClassifier(
    objective="binary",
    scale_pos_weight=pos_weight,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    random_state=42
)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50)]
)

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

# Probabilités calibrées (important pour le Brier Score)
proba = model.predict_proba(X_test)[:, 1]

bs = brier_score_loss(y_test, proba)
print(f"Brier Score : {bs:.4f}")
# Référence : modèle nul (prédit toujours la moyenne) → ~0.02-0.05
# Objectif   : < 0.01

# Brier Score par aéroport pour détecter les disparités
for airport in df_test["airport"].unique():
    mask = df_test["airport"] == airport
    bs_ap = brier_score_loss(y_test[mask], proba[mask])
    print(f"  {airport:10s} : {bs_ap:.4f}")                                                                            

In [55]:
"""
Survival Analysis Pipeline — Data Battle 2026
==============================================
Utilise un modèle Weibull AFT pour prédire la fin des alertes orageuses.

Depuis la courbe S(t | features), on extrait directement :
  - predicted_date_end_alert  (médiane de la distribution)
  - confidence                (1 - S(30 min))

Ces deux outputs sont mécaniquement cohérents car issus de la même courbe.

Dépendances :
    pip install lifelines pandas numpy
"""

import pandas as pd
import numpy as np
from lifelines import WeibullAFTFitter
from lifelines.utils import concordance_index




# ─────────────────────────────────────────────────────────────────────────────
# CONSTRUCTION DU DATASET DE SURVIE
# ─────────────────────────────────────────────────────────────────────────────

def build_survival_dataset(df_model: pd.DataFrame) -> pd.DataFrame:
    """
    Transforme le dataset de features en dataset de survie.

    Chaque ligne représente un instant d'observation dans une alerte.
    La variable de durée est `remaining_minutes` — le temps restant
    jusqu'au dernier CG réel.

    Paramètre
    ---------
    df_model : issu de build_model_dataset(), doit contenir
               `remaining_minutes` et toutes les features.

    Retourne
    --------
    df_surv : même dataset avec la colonne `event` ajoutée.
              event = 1 pour toutes les lignes (pas de censure dans
              ce dataset — tous les orages ont une fin observée).
    """
    df_surv = df_model.copy()

    # Durée de survie : temps restant jusqu'à la fin de l'alerte
    # On s'assure qu'il n'y a pas de valeurs nulles ou négatives
    df_surv["duration"] = df_surv["remaining_minutes"].clip(lower=0.1)

    # Tous les épisodes ont une fin observée → event = 1 partout
    df_surv["event"] = 1

    # On supprime les lignes avec duration = 0 (instant = fin exacte)
    df_surv = df_surv[df_surv["duration"] > 0].copy()

    print(f"Dataset survie : {len(df_surv)} observations")
    print(f"Durée médiane restante : {df_surv['duration'].median():.1f} min")
    print(f"Durée max restante     : {df_surv['duration'].max():.1f} min")

    return df_surv


# ─────────────────────────────────────────────────────────────────────────────
# ENTRAÎNEMENT DU MODÈLE WEIBULL AFT
# ─────────────────────────────────────────────────────────────────────────────

def train_weibull(
    df_surv_train: pd.DataFrame,
    feature_cols: list[str],
    penalizer: float = 0.1
) -> WeibullAFTFitter:
    """
    Entraîne un modèle Weibull Accelerated Failure Time.

    Le modèle Weibull AFT paramétrise directement log(T) :
        log(T) = μ + γ·X + σ·ε

    → Il prédit la distribution complète du temps restant,
      pas juste un point.

    Paramètres
    ----------
    df_surv_train : dataset de survie (train)
    feature_cols  : liste des features à utiliser
    penalizer     : régularisation L2 (évite le surapprentissage)
    """
    print("Entraînement Weibull AFT...")

    # Colonnes nécessaires : features + duration + event
    cols = feature_cols + ["duration", "event"]
    df_fit = df_surv_train[cols].dropna()

    model = WeibullAFTFitter(penalizer=penalizer)
    model.fit(
        df_fit,
        duration_col="duration",
        event_col="event"
    )

    print("✓ Modèle entraîné.")
    print(f"  Log-likelihood : {model.log_likelihood_:.2f}")

    return model


# ─────────────────────────────────────────────────────────────────────────────
# ÉVALUATION DU MODÈLE
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_model(
    model: WeibullAFTFitter,
    df_surv_test: pd.DataFrame,
    feature_cols: list[str],
    horizon_minutes: int = 30
) -> dict:
    """
    Évalue le modèle sur le jeu de test.

    Métriques calculées :
    - MAE sur la durée restante prédite (médiane)
    - C-index (concordance — équivalent AUC pour la survie)
    - Calibration du confidence score à l'horizon donné

    Retourne un dict de métriques.
    """
    df_eval = df_surv_test[feature_cols + ["duration", "event"]].dropna()

    # Médiane prédite (temps restant estimé)
    predicted_median = model.predict_median(df_eval[feature_cols])

    # MAE sur la durée
    mae = (predicted_median - df_eval["duration"]).abs().mean()

    # C-index : est-ce que le modèle ordonne correctement les durées ?
    ci = concordance_index(
        df_eval["duration"],
        -predicted_median,
        df_eval["event"]
    )

    # Courbes de survie à l'horizon (confidence = 1 - S(horizon))
    surv_at_horizon = model.predict_survival_function(
        df_eval[feature_cols],
        times=[horizon_minutes]
    ).T.values.flatten()
    confidence = 1 - surv_at_horizon

    # Brier Score à l'horizon
    # y_true = 1 si l'alerte est déjà terminée à l'horizon (duration < horizon)
    y_true = (df_eval["duration"] <= horizon_minutes).astype(int).values
    brier = np.mean((confidence - y_true) ** 2)

    metrics = {
        "MAE_minutes":   round(mae, 2),
        "C_index":       round(ci, 4),
        "Brier_score":   round(brier, 4),
        "N_test":        len(df_eval)
    }

    print("\n── Évaluation ──────────────────────────────")
    for k, v in metrics.items():
        print(f"  {k:20s} : {v}")

    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# GÉNÉRATION DES PRÉDICTIONS AU FORMAT DU JURY
# ─────────────────────────────────────────────────────────────────────────────

def generate_predictions(
    model: WeibullAFTFitter,
    df_test: pd.DataFrame,
    feature_cols: list[str],
    horizon_minutes: int = 30,
    output_path: str = "predictions.csv"
) -> pd.DataFrame:
    """
    Génère le fichier predictions.csv attendu par le jury.

    Format de sortie (une ligne par instant d'observation) :
    ┌──────────────────────────────────────────────────────────────────┐
    │ airport | airport_alert_id | prediction_date                    │
    │ predicted_date_end_alert   | confidence                         │
    └──────────────────────────────────────────────────────────────────┘

    Les deux outputs sont extraits de la même courbe S(t) :
      - predicted_date_end_alert = obs_time + médiane(distribution)
      - confidence               = 1 - S(horizon_minutes)

    Paramètres
    ----------
    model           : modèle Weibull AFT entraîné
    df_test         : dataset de test (avec obs_time, airport, alert_id)
    feature_cols    : features utilisées à l'entraînement
    horizon_minutes : horizon pour le calcul de confiance (défaut 30 min)
    output_path     : chemin du fichier CSV de sortie
    """
    print("Génération des prédictions...")

    df_valid = df_test.dropna(subset=feature_cols).copy()
    X = df_valid[feature_cols]

    # ── Output 1 : temps restant prédit (médiane de la distribution) ──────────
    remaining_pred = model.predict_median(X).values
    # Clip pour éviter les prédictions négatives
    remaining_pred = np.clip(remaining_pred, 0, None)

    predicted_end = (
        df_valid["obs_time"].values
        + pd.to_timedelta(remaining_pred, unit="m")
    )

    # ── Output 2 : confidence = 1 - S(horizon) ────────────────────────────────
    surv_at_horizon = model.predict_survival_function(
        X, times=[horizon_minutes]
    ).T.values.flatten()
    confidence = 1 - surv_at_horizon

    # ── Assemblage ────────────────────────────────────────────────────────────
    predictions = pd.DataFrame({
        "airport":                  df_valid["airport"].values,
        "airport_alert_id":         df_valid["alert_id"].values,
        "prediction_date":          df_valid["obs_time"].values,
        "predicted_date_end_alert": predicted_end,
        "confidence":               confidence
    })

    predictions.to_csv(output_path, index=False)
    print(f"✓ {len(predictions)} prédictions sauvegardées → {output_path}")
    print(f"  Confiance moyenne    : {confidence.mean():.3f}")
    print(f"  Temps restant médian : {np.median(remaining_pred):.1f} min")

    return predictions


# ─────────────────────────────────────────────────────────────────────────────
# ÉVALUATION AU FORMAT DU JURY (Gain & Risque)
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_jury_metrics(
    predictions: pd.DataFrame,
    df_raw: pd.DataFrame,
    n_thetas: int = 20,
    min_dist_km: float = 3.0,
    acceptable_risk: float = 0.02,
    max_gap_minutes: int = 30
) -> dict:
    """
    Reproduit exactement la logique d'évaluation du jury.

    Pour chaque valeur de θ entre 0 et 1 :
      1. Garde les prédictions avec confidence >= θ
      2. Prend la predicted_date_end_alert la plus précoce par alerte
      3. Calcule le gain vs baseline (last_cg + 30 min)
      4. Calcule le risque (éclairs dangereux manqués)

    Retourne le θ optimal et les métriques associées.
    """
    df_raw = df_raw.copy()
    df_raw["date"] = pd.to_datetime(df_raw["date"], utc=True)
    predictions["predicted_date_end_alert"] = pd.to_datetime(
        predictions["predicted_date_end_alert"], utc=True
    )

    # Éclairs dangereux totaux (dist < 3 km)
    tot_dangerous = len(df_raw[df_raw["dist"] < min_dist_km])
    alerts = df_raw.groupby(["airport", "airport_alert_id"])

    thetas = [i / n_thetas for i in range(n_thetas)]
    results = {}

    for theta in thetas:
        # Prédictions retenues pour ce θ
        retained = predictions[predictions["confidence"] >= theta]
        if len(retained) == 0:
            continue

        # Prédiction la plus précoce par alerte
        earliest = retained.groupby(
            ["airport", "airport_alert_id"]
        )["predicted_date_end_alert"].min()

        gain, missed = 0, 0

        for (airport, alert_id), end_pred in earliest.items():
            try:
                group = alerts.get_group((airport, alert_id))
            except KeyError:
                continue

            dates = pd.to_datetime(group["date"], utc=True)

            # Baseline : dernier éclair + 30 minutes
            baseline = dates.max() + pd.Timedelta(minutes=max_gap_minutes)

            # Gain en secondes vs baseline
            gain += (baseline - end_pred).total_seconds()

            # Éclairs dangereux manqués (tombent après la levée d'alerte prédite)
            dangerous = group[group["dist"] < min_dist_km]
            dangerous_dates = pd.to_datetime(dangerous["date"], utc=True)
            missed += (dangerous_dates > end_pred).sum()

        risk = missed / tot_dangerous if tot_dangerous > 0 else 0
        results[theta] = {"gain_seconds": gain, "risk": risk, "missed": missed}

    # Sélection du θ optimal : max gain avec risk < acceptable_risk
    valid = {
        theta: v for theta, v in results.items()
        if v["risk"] < acceptable_risk
    }

    if not valid:
        print("⚠️  Aucun θ ne respecte la contrainte de risque.")
        return {}

    best_theta = max(valid, key=lambda t: valid[t]["gain_seconds"])
    best = valid[best_theta]

    print("\n── Métriques jury ──────────────────────────")
    print(f"  Risque acceptable    : {acceptable_risk}")
    print(f"  θ optimal            : {best_theta:.2f}")
    print(f"  Gain de temps        : {best['gain_seconds']/3600:.1f} heures")
    print(f"  Risque               : {best['risk']:.4f}")
    print(f"  Éclairs manqués      : {best['missed']}")

    return {
        "best_theta":    best_theta,
        "gain_hours":    best["gain_seconds"] / 3600,
        "risk":          best["risk"],
        "missed":        best["missed"],
        "all_results":   results
    }


# ─────────────────────────────────────────────────────────────────────────────
# POINT D'ENTRÉE
# ─────────────────────────────────────────────────────────────────────────────



In [56]:
 # Fichier d'entraînement (2016-2022)
df_alert_train, df_context_train = load_and_prepare("segment_alerts_all_airports_train.csv")
df_train = build_model_dataset(df_alert_train, df_context_train, step_minutes=5)

# Fichier de test (2023-2025)
df_alert_test, df_context_test = load_and_prepare("dataset_set.csv")
df_test = build_model_dataset(df_alert_test, df_context_test, step_minutes=5)

  Traitement alerte 2601/2627...
Dataset construit : 14877 observations.
  Traitement alerte 501/577...
Dataset construit : 3935 observations.


In [57]:
# ── 4. Dataset de survie ──────────────────────────────────────────────────
print("\n4. Préparation dataset survie...")
df_surv_train = build_survival_dataset(df_train)
df_surv_test  = build_survival_dataset(df_test)


4. Préparation dataset survie...
Dataset survie : 14877 observations
Durée médiane restante : 36.5 min
Durée max restante     : 573.7 min
Dataset survie : 3935 observations
Durée médiane restante : 27.8 min
Durée max restante     : 273.2 min


In [58]:
# ── 5. Features à utiliser ────────────────────────────────────────────────
exclude_cols = {
    "airport", "alert_id", "obs_time", "year",
    "y", "remaining_minutes", "last_cg_time",
    "duration", "event"
}
feature_cols = [c for c in df_surv_train.columns if c not in exclude_cols]
print(f"\n5. {len(feature_cols)} features utilisées.")


5. 43 features utilisées.


In [59]:
# ── 6. Entraînement ───────────────────────────────────────────────────────
print("\n6. Entraînement...")
model = train_weibull(df_surv_train, feature_cols, penalizer=1)


6. Entraînement...
Entraînement Weibull AFT...
✓ Modèle entraîné.
  Log-likelihood : -54931.43


/home/safsaf/miniconda3/lib/python3.13/site-packages/lifelines/fitters/__init__.py:2090: ApproximationWarning: The Hessian was not invertible. We will instead approximate it using the pseudo-inverse.

It's advisable to not trust the variances reported, and to be suspicious of the fitted parameters too.

Some ways to possible ways fix this:

0. Are there any lifelines warnings outputted during the `fit`?
1. Does a particularly large variable need to be centered to 0?
2. Inspect your DataFrame: does everything look as expected? Do you need to add/drop a constant (intercept) column?
3. Is there high-collinearity in the dataset? Try using the variance inflation factor (VIF) to find redundant variables.
4. Trying adding a small penalizer (or changing it, if already present). Example: `WeibullAFTFitter(penalizer=0.01).fit(...)`.
5. Are there any extreme outliers? Try modeling them or dropping them to see if it helps convergence.

  warnings.warn(warning_text, exceptions.ApproximationWarning)

In [63]:
# ── 7. Évaluation technique ───────────────────────────────────────────────
print("\n7. Évaluation technique...")
metrics = evaluate_model(model, df_surv_test, feature_cols)


7. Évaluation technique...

── Évaluation ──────────────────────────────
  MAE_minutes          : 29.08
  C_index              : 0.4565
  Brier_score          : 0.2542
  N_test               : 3935


In [62]:
# ── Remplissage des NaN ───────────────────────────────────────────────────────

# Distances : NaN = aucun éclair dans la fenêtre = orage absent → valeur max de zone
dist_cols = [c for c in feature_cols if "dist" in c]
df_surv_train[dist_cols] = df_surv_train[dist_cols].fillna(30)
df_surv_test[dist_cols]  = df_surv_test[dist_cols].fillna(30)

# Ratios et tendances : NaN = pas d'activité → 0
zero_cols = [c for c in feature_cols if any(k in c for k in ["ratio", "trend"])]
df_surv_train[zero_cols] = df_surv_train[zero_cols].fillna(0)
df_surv_test[zero_cols]  = df_surv_test[zero_cols].fillna(0)

# Vérification — il ne doit plus rester de NaN
remaining_nan = df_surv_train[feature_cols].isna().sum().sum()
print(f"NaN restants : {remaining_nan}")  # doit afficher 0

# ── Standardisation ───────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

# On ne standardise pas les features cycliques (déjà entre -1 et 1)
cyclic_cols = ["sin_hour", "cos_hour", "sin_month", "cos_month"]
cols_to_scale = [c for c in feature_cols if c not in cyclic_cols]

scaler = StandardScaler()
df_surv_train[cols_to_scale] = scaler.fit_transform(df_surv_train[cols_to_scale])
df_surv_test[cols_to_scale]  = scaler.transform(df_surv_test[cols_to_scale])

# ── Ré-entraînement avec pénaliseur plus fort ─────────────────────────────────
model = train_weibull(df_surv_train, feature_cols, penalizer=1.0)

NaN restants : 0
Entraînement Weibull AFT...
✓ Modèle entraîné.
  Log-likelihood : -74509.13


/home/safsaf/miniconda3/lib/python3.13/site-packages/lifelines/fitters/__init__.py:2090: ApproximationWarning: The Hessian was not invertible. We will instead approximate it using the pseudo-inverse.

It's advisable to not trust the variances reported, and to be suspicious of the fitted parameters too.

Some ways to possible ways fix this:

0. Are there any lifelines warnings outputted during the `fit`?
1. Does a particularly large variable need to be centered to 0?
2. Inspect your DataFrame: does everything look as expected? Do you need to add/drop a constant (intercept) column?
3. Is there high-collinearity in the dataset? Try using the variance inflation factor (VIF) to find redundant variables.
4. Trying adding a small penalizer (or changing it, if already present). Example: `WeibullAFTFitter(penalizer=0.01).fit(...)`.
5. Are there any extreme outliers? Try modeling them or dropping them to see if it helps convergence.

  warnings.warn(warning_text, exceptions.ApproximationWarning)

In [61]:
# Diagnostic 1 — taille du dataset
print("Shape df_train :", df_train.shape)
print("Alertes train  :", df_train["alert_id"].nunique())

# Diagnostic 2 — NaN dans les features
print("\nNaN par feature :")
print(df_surv_train[feature_cols].isna().sum().sort_values(ascending=False).head(15))

# Diagnostic 3 — distribution de remaining_minutes
print("\nDistribution remaining_minutes :")
print(df_surv_train["duration"].describe())

Shape df_train : (14877, 50)
Alertes train  : 682

NaN par feature :
mean_dist_cg_5min        4107
max_dist_cg_5min         4107
std_dist_cg_5min         4107
dist_trend_5_30          4107
ratio_ctx_alert_5min     2085
max_dist_cg_15min         863
std_dist_cg_15min         863
mean_dist_cg_15min        863
ratio_ctx_alert_15min     283
minutes_since_last_ic       0
t_elapsed_min               0
mean_abs_amp_cg_5min        0
std_abs_amp_cg_5min         0
total_count_5min            0
cg_ctx_count_5min           0
dtype: int64

Distribution remaining_minutes :
count    14877.000000
mean        57.822313
std         66.181154
min          0.100000
25%         14.366667
50%         36.533333
75%         77.433333
max        573.716667
Name: duration, dtype: float64


In [50]:
# ── 8. Génération des prédictions ─────────────────────────────────────────
print("\n8. Génération des prédictions...")
predictions = generate_predictions(
    model, df_test, feature_cols, output_path="predictions.csv"
)



8. Génération des prédictions...
Génération des prédictions...


OutOfBoundsDatetime: cannot convert input 1371140116.2011082 with the unit 'm'

In [ ]:
# ── 9. Évaluation au format jury ──────────────────────────────────────────
print("\n9. Évaluation jury (Gain & Risque)...")
import pandas as pd
df_raw = pd.read_csv(CSV_PATH)
jury_metrics = evaluate_jury_metrics(predictions, df_raw)

In [37]:
def compute_features_at_time_clean(
    grp_alert: pd.DataFrame,
    grp_context: pd.DataFrame,
    t: pd.Timestamp
) -> dict:

    features = {}

    # ─────────────────────────────────────────────
    # DATA PASSÉE
    # ─────────────────────────────────────────────
    past_alert = grp_alert[grp_alert["date"] < t]
    past_cg = past_alert[~past_alert["icloud"]]

    # ─────────────────────────────────────────────
    # TEMPS DEPUIS DERNIER CG
    # ─────────────────────────────────────────────
    if len(past_cg) > 0:
        last_cg_time = past_cg["date"].max()
        features["minutes_since_last_cg"] = (t - last_cg_time).total_seconds() / 60
    else:
        features["minutes_since_last_cg"] = 999.0  # gros = aucun signal

    # ─────────────────────────────────────────────
    # BINS TEMPORELS (non cumulés)
    # ─────────────────────────────────────────────
    bins = [(0, 5), (5, 10), (10, 15)]

    for start, end in bins:
        t_start = t - pd.Timedelta(minutes=end)
        t_end   = t - pd.Timedelta(minutes=start)

        window = past_cg[
            (past_cg["date"] >= t_start) &
            (past_cg["date"] < t_end)
        ]

        key = f"{start}_{end}"

        # Comptage
        features[f"cg_count_{key}"] = len(window)

        # Amplitude
        if len(window) > 0:
            features[f"mean_amp_{key}"] = window["amplitude"].abs().mean()
        else:
            features[f"mean_amp_{key}"] = 0.0

        # Distance
        if len(window) > 0:
            features[f"mean_dist_{key}"] = window["dist"].mean()
        else:
            features[f"mean_dist_{key}"] = 0.0

    # ─────────────────────────────────────────────
    # CONTEXTE (extérieur 20-30 km)
    # ─────────────────────────────────────────────
    past_ctx = grp_context[
        (grp_context["date"] < t) &
        (grp_context["date"] >= t - pd.Timedelta(minutes=15))
    ]
    past_ctx_cg = past_ctx[~past_ctx["icloud"]]

    features["cg_ctx_15min"] = len(past_ctx_cg)

    # ─────────────────────────────────────────────
    # TEMPS CYCLIQUE (optionnel mais safe)
    # ─────────────────────────────────────────────
    features["sin_hour"] = np.sin(2 * np.pi * t.hour / 24)
    features["cos_hour"] = np.cos(2 * np.pi * t.hour / 24)

    return features

In [40]:
print(df_surv_train.head())

   airport  alert_id                  obs_time  year  \
0  Ajaccio       1.0 2016-01-02 21:27:53+00:00  2016   
1  Ajaccio       6.0 2016-02-25 14:15:03+00:00  2016   
2  Ajaccio       6.0 2016-02-25 14:20:03+00:00  2016   
3  Ajaccio       6.0 2016-02-25 14:25:03+00:00  2016   
4  Ajaccio       6.0 2016-02-25 14:30:03+00:00  2016   

               last_cg_time  remaining_minutes  y  t_elapsed_min  \
0 2016-01-02 21:28:54+00:00           1.016667  0            5.0   
1 2016-02-25 15:02:18+00:00          47.250000  0            5.0   
2 2016-02-25 15:02:18+00:00          42.250000  0           10.0   
3 2016-02-25 15:02:18+00:00          37.250000  0           15.0   
4 2016-02-25 15:02:18+00:00          32.250000  0           20.0   

   minutes_since_last_cg  minutes_since_last_ic  ...  dist_trend_5_30  \
0               0.816667                    5.0  ...         0.000000   
1               2.066667                    5.0  ...         0.000000   
2               0.283333           

In [51]:
from sklearn.preprocessing import StandardScaler
feature_cols = [
    "minutes_since_last_cg",
    "cg_count_0_5",
    "cg_count_5_10",
    "cg_count_10_15",
    "mean_amp_0_5",
    "mean_amp_5_10",
    "mean_amp_10_15",
    "mean_dist_0_5",
    "mean_dist_5_10",
    "mean_dist_10_15",
    "cg_ctx_15min",
    "sin_hour",
    "cos_hour"
]
missing = [c for c in feature_cols if c not in df_surv_train.columns]
print("Missing:", missing)
scaler = StandardScaler()
df_surv_train[feature_cols] = scaler.fit_transform(df_surv_train[feature_cols])
df_surv_test[feature_cols]  = scaler.transform(df_surv_test[feature_cols])

Missing: []


In [32]:
# ─────────────────────────────────────────────────────────────────────────────
# CHARGEMENT ET PRÉPARATION
# ─────────────────────────────────────────────────────────────────────────────

def load_and_prepare(csv_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Charge le CSV brut, parse les dates, filtre Pise 2016 IC,
    et sépare zone alerte (< 20 km) et zone contexte (20-30 km).

    Retourne : (df_alert, df_context)
    """
    df = pd.read_csv(csv_path)

    # Parsing de la date
    df["date"] = pd.to_datetime(df["date"], utc=True)

    # Zone alerte : éclairs avec une alerte assignée (dist < 20 km)
    df_alert = df[df["airport_alert_id"].notna()].copy()

    # Zone contexte : éclairs entre 20 et 30 km (pas d'alerte assignée)
    df_context = df[
        (df["dist"] > 20) & (df["dist"] <= 30)
    ].copy()

    return df_alert, df_context

def compute_features_at_time_clean(
    grp_alert: pd.DataFrame,
    grp_context: pd.DataFrame,
    t: pd.Timestamp
) -> dict:

    features = {}

    # ─────────────────────────────────────────────
    # DATA PASSÉE
    # ─────────────────────────────────────────────
    past_alert = grp_alert[grp_alert["date"] < t]
    past_cg = past_alert[~past_alert["icloud"]]

    # ─────────────────────────────────────────────
    # TEMPS DEPUIS DERNIER CG
    # ─────────────────────────────────────────────
    if len(past_cg) > 0:
        last_cg_time = past_cg["date"].max()
        features["minutes_since_last_cg"] = (t - last_cg_time).total_seconds() / 60
    else:
        features["minutes_since_last_cg"] = 999.0  # gros = aucun signal

    # ─────────────────────────────────────────────
    # BINS TEMPORELS (non cumulés)
    # ─────────────────────────────────────────────
    bins = [(0, 5), (5, 10), (10, 15)]

    for start, end in bins:
        t_start = t - pd.Timedelta(minutes=end)
        t_end   = t - pd.Timedelta(minutes=start)

        window = past_cg[
            (past_cg["date"] >= t_start) &
            (past_cg["date"] < t_end)
        ]

        key = f"{start}_{end}"

        # Comptage
        features[f"cg_count_{key}"] = len(window)

        # Amplitude
        if len(window) > 0:
            features[f"mean_amp_{key}"] = window["amplitude"].abs().mean()
        else:
            features[f"mean_amp_{key}"] = 0.0

        # Distance
        if len(window) > 0:
            features[f"mean_dist_{key}"] = window["dist"].mean()
        else:
            features[f"mean_dist_{key}"] = 0.0

    # ─────────────────────────────────────────────
    # CONTEXTE (extérieur 20-30 km)
    # ─────────────────────────────────────────────
    past_ctx = grp_context[
        (grp_context["date"] < t) &
        (grp_context["date"] >= t - pd.Timedelta(minutes=15))
    ]
    past_ctx_cg = past_ctx[~past_ctx["icloud"]]

    features["cg_ctx_15min"] = len(past_ctx_cg)

    # ─────────────────────────────────────────────
    # TEMPS CYCLIQUE (optionnel mais safe)
    # ─────────────────────────────────────────────
    features["sin_hour"] = np.sin(2 * np.pi * t.hour / 24)
    features["cos_hour"] = np.cos(2 * np.pi * t.hour / 24)

    return features

# ─────────────────────────────────────────────────────────────────────────────
# CONSTRUCTION DU DATASET COMPLET
# ─────────────────────────────────────────────────────────────────────────────
 
def build_model_dataset(
    df_alert: pd.DataFrame,
    df_context: pd.DataFrame,
    step_minutes: int = 5,
    horizon_minutes: int = 30
) -> pd.DataFrame:
    """
    Génère le dataset de modélisation.
 
    Pour chaque alerte, on observe toutes les `step_minutes` minutes.
    À chaque instant t :
      - y = 1 si aucun CG dans les `horizon_minutes` suivants (fin imminente)
      - y = 0 sinon
 
    Paramètres
    ----------
    df_alert        : zone alerte (dist < 20 km), issu de load_and_prepare()
    df_context      : zone contexte (20-30 km), issu de load_and_prepare()
    step_minutes    : pas d'échantillonnage temporel (défaut : 5 min)
    horizon_minutes : horizon de prédiction pour la target (défaut : 30 min)
    """
    records = []
    groups = list(df_alert.groupby(["airport", "airport_alert_id"]))
    total = len(groups)
 
    for i, ((airport, alert_id), grp) in enumerate(groups):
        if i % 100 == 0:
            print(f"  Traitement alerte {i+1}/{total}...", end="\r")
 
        grp = grp.sort_values("date")
        grp_cg = grp[~grp["icloud"]]
 
        # On ignore les alertes sans aucun CG (pas de target possible)
        if len(grp_cg) == 0:
            continue
 
        alert_start  = grp["date"].min()
        last_cg_time = grp_cg["date"].max()  # fin réelle de l'alerte
 
        # Contexte : même aéroport seulement
        grp_ctx = df_context[df_context["airport"] == airport].copy()
 
        # Fenêtres d'observation : de step_minutes après le début jusqu'à la fin
        obs_times = pd.date_range(
            start=alert_start + pd.Timedelta(minutes=step_minutes),
            end=last_cg_time,
            freq=f"{step_minutes}min"
        )
 
        for t in obs_times:
            # ── Target ────────────────────────────────────────────────────────
            future_cg = grp_cg[
                (grp_cg["date"] > t) &
                (grp_cg["date"] <= t + pd.Timedelta(minutes=horizon_minutes))
            ]
            y = 1 if len(future_cg) == 0 else 0
 
            # ── Features ──────────────────────────────────────────────────────
            feats = compute_features_at_time(
                grp_alert=grp,
                grp_context=grp_ctx,
                t=t
            )
 
            records.append({
                "airport":           airport,
                "alert_id":          alert_id,
                "obs_time":          t,
                "year":              t.year,
                "last_cg_time":      last_cg_time,
                "remaining_minutes": (last_cg_time - t).total_seconds() / 60,
                "y":                 y,
                **feats
            })
 
    print(f"\nDataset construit : {len(records)} observations.")
    return pd.DataFrame(records)

In [47]:
import pandas as pd
import numpy as np
from lifelines import WeibullAFTFitter
from sklearn.preprocessing import StandardScaler

# ─────────────────────────────────────────────
# LOAD
# ─────────────────────────────────────────────

def load_and_prepare(csv_path):
    df = pd.read_csv(csv_path)
    df["date"] = pd.to_datetime(df["date"], utc=True)

    df_alert = df[df["airport_alert_id"].notna()].copy()
    df_context = df[(df["dist"] > 20) & (df["dist"] <= 30)].copy()

    return df_alert, df_context


# ─────────────────────────────────────────────
# FEATURES CLEAN
# ─────────────────────────────────────────────

def compute_features_at_time_clean(grp_alert, grp_context, t):
    features = {}

    past_alert = grp_alert[grp_alert["date"] < t]
    past_cg = past_alert[~past_alert["icloud"]]

    # Temps depuis dernier CG
    if len(past_cg) > 0:
        last_cg = past_cg["date"].max()
        features["minutes_since_last_cg"] = (t - last_cg).total_seconds() / 60
    else:
        features["minutes_since_last_cg"] = 999.0

    # Bins 5 min non cumulés
    bins = [(0,5), (5,10), (10,15)]

    for start, end in bins:
        t_start = t - pd.Timedelta(minutes=end)
        t_end   = t - pd.Timedelta(minutes=start)

        window = past_cg[
            (past_cg["date"] >= t_start) &
            (past_cg["date"] < t_end)
        ]

        key = f"{start}_{end}"

        features[f"cg_count_{key}"] = len(window)
        features[f"mean_amp_{key}"] = window["amplitude"].abs().mean() if len(window) > 0 else 0.0
        features[f"mean_dist_{key}"] = window["dist"].mean() if len(window) > 0 else 0.0

    # Contexte 15 min
    past_ctx = grp_context[
        (grp_context["date"] < t) &
        (grp_context["date"] >= t - pd.Timedelta(minutes=15))
    ]
    past_ctx_cg = past_ctx[~past_ctx["icloud"]]

    features["cg_ctx_15min"] = len(past_ctx_cg)

    # Temps cyclique
    features["sin_hour"] = np.sin(2 * np.pi * t.hour / 24)
    features["cos_hour"] = np.cos(2 * np.pi * t.hour / 24)

    return features


# ─────────────────────────────────────────────
# BUILD DATASET
# ─────────────────────────────────────────────

def build_model_dataset(df_alert, df_context, step_minutes=5):
    records = []

    for (airport, alert_id), grp in df_alert.groupby(["airport", "airport_alert_id"]):
        grp = grp.sort_values("date")
        grp_cg = grp[~grp["icloud"]]

        if len(grp_cg) == 0:
            continue

        alert_start = grp["date"].min()
        last_cg = grp_cg["date"].max()

        grp_ctx = df_context[df_context["airport"] == airport]

        obs_times = pd.date_range(
            start=alert_start + pd.Timedelta(minutes=step_minutes),
            end=last_cg,
            freq=f"{step_minutes}min"
        )

        for t in obs_times:
            feats = compute_features_at_time_clean(grp, grp_ctx, t)

            records.append({
                "airport": airport,
                "alert_id": alert_id,
                "obs_time": t,
                "remaining_minutes": (last_cg - t).total_seconds() / 60,
                **feats
            })

    return pd.DataFrame(records)


# ─────────────────────────────────────────────
# SURVIVAL DATASET
# ─────────────────────────────────────────────

def build_survival_dataset(df):
    df = df.copy()
    df["duration"] = df["remaining_minutes"].clip(lower=0.1)
    df["event"] = 1
    return df


# ─────────────────────────────────────────────
# PIPELINE
# ─────────────────────────────────────────────

# LOAD
df_alert_train, df_context_train = load_and_prepare("segment_alerts_all_airports_train.csv")
df_alert_test, df_context_test = load_and_prepare("dataset_set.csv")

# BUILD
df_train = build_model_dataset(df_alert_train, df_context_train)
df_test  = build_model_dataset(df_alert_test, df_context_test)

# SURVIVAL
df_surv_train = build_survival_dataset(df_train)
df_surv_test  = build_survival_dataset(df_test)

# FEATURES
feature_cols = [
    "minutes_since_last_cg",
    "cg_count_0_5", "cg_count_5_10", "cg_count_10_15",
    "mean_amp_0_5", "mean_amp_5_10", "mean_amp_10_15",
    "mean_dist_0_5", "mean_dist_5_10", "mean_dist_10_15",
    "cg_ctx_15min",
    "sin_hour", "cos_hour"
]

# CLEAN NaN
df_surv_train[feature_cols] = df_surv_train[feature_cols].fillna(0)
df_surv_test[feature_cols]  = df_surv_test[feature_cols].fillna(0)

# SCALE
scaler = StandardScaler()
df_surv_train[feature_cols] = scaler.fit_transform(df_surv_train[feature_cols])
df_surv_test[feature_cols]  = scaler.transform(df_surv_test[feature_cols])

# MODEL
model = WeibullAFTFitter(penalizer=1.0)
model.fit(df_surv_train[feature_cols + ["duration", "event"]],
          duration_col="duration",
          event_col="event")

print("Model trained.")

Model trained.


In [52]:
import pandas as pd
import numpy as np
from lifelines import WeibullAFTFitter
from sklearn.preprocessing import StandardScaler

# ─────────────────────────────────────────────
# LOAD
# ─────────────────────────────────────────────

def load_and_prepare(csv_path):
    df = pd.read_csv(csv_path)
    df["date"] = pd.to_datetime(df["date"], utc=True)
    df_alert = df[df["airport_alert_id"].notna()].copy()
    df_context = df[(df["dist"] > 20) & (df["dist"] <= 30)].copy()
    return df_alert, df_context

# ─────────────────────────────────────────────
# FEATURES CUMULATIVES
# ─────────────────────────────────────────────

def compute_cumulative_features(grp_alert, grp_context, t, window_minutes=10):
    features = {}
    
    # alert CG jusqu'au temps t
    past_alert = grp_alert[grp_alert["date"] <= t]
    past_cg = past_alert[~past_alert["icloud"]]

    # temps depuis dernier CG
    if len(past_cg) > 0:
        last_cg = past_cg["date"].max()
        features["minutes_since_last_cg"] = (t - last_cg).total_seconds() / 60
    else:
        features["minutes_since_last_cg"] = 999.0

    # Fenêtre cumulative de 10 min avant t
    t_start = t - pd.Timedelta(minutes=window_minutes)
    window = past_cg[(past_cg["date"] >= t_start) & (past_cg["date"] <= t)]
    
    features["cg_count_10min"] = len(window)
    features["mean_amp_10min"] = window["amplitude"].abs().mean() if len(window) > 0 else 0.0
    features["mean_dist_10min"] = window["dist"].mean() if len(window) > 0 else 0.0

    # Contexte 15 min
    past_ctx = grp_context[(grp_context["date"] <= t) &
                           (grp_context["date"] >= t - pd.Timedelta(minutes=15))]
    past_ctx_cg = past_ctx[~past_ctx["icloud"]]
    features["cg_ctx_15min"] = len(past_ctx_cg)

    # Temps cyclique
    features["sin_hour"] = np.sin(2 * np.pi * t.hour / 24)
    features["cos_hour"] = np.cos(2 * np.pi * t.hour / 24)

    return features

# ─────────────────────────────────────────────
# BUILD DATASET CUMULATIF
# ─────────────────────────────────────────────

def build_model_dataset_cumulative(df_alert, df_context, step_minutes=5, window_minutes=10):
    records = []
    
    for (airport, alert_id), grp in df_alert.groupby(["airport", "airport_alert_id"]):
        grp = grp.sort_values("date")
        grp_cg = grp[~grp["icloud"]]
        if len(grp_cg) == 0:
            continue

        alert_start = grp["date"].min()
        last_cg_time = grp_cg["date"].max()
        grp_ctx = df_context[df_context["airport"] == airport]

        obs_times = pd.date_range(
            start=alert_start + pd.Timedelta(minutes=step_minutes),
            end=last_cg_time,
            freq=f"{step_minutes}min"
        )

        for t in obs_times:
            feats = compute_cumulative_features(grp, grp_ctx, t, window_minutes)
            records.append({
                "airport": airport,
                "alert_id": alert_id,
                "obs_time": t,
                "remaining_minutes": (last_cg_time - t).total_seconds() / 60,
                **feats
            })

    return pd.DataFrame(records)

# ─────────────────────────────────────────────
# SURVIVAL DATASET
# ─────────────────────────────────────────────

def build_survival_dataset(df):
    df = df.copy()
    df["duration"] = df["remaining_minutes"].clip(lower=0.1)
    df["event"] = 1
    return df

# ─────────────────────────────────────────────
# PIPELINE
# ─────────────────────────────────────────────

# LOAD
df_alert_train, df_context_train = load_and_prepare("segment_alerts_all_airports_train.csv")
df_alert_test, df_context_test = load_and_prepare("dataset_set.csv")

# BUILD
df_train = build_model_dataset_cumulative(df_alert_train, df_context_train,
                                          step_minutes=5, window_minutes=10)
df_test  = build_model_dataset_cumulative(df_alert_test, df_context_test,
                                          step_minutes=5, window_minutes=10)

# SURVIVAL
df_surv_train = build_survival_dataset(df_train)
df_surv_test  = build_survival_dataset(df_test)

# FEATURES
feature_cols = [
    "minutes_since_last_cg",
    "cg_count_10min", "mean_amp_10min", "mean_dist_10min",
    "cg_ctx_15min",
    "sin_hour", "cos_hour"
]

# CLEAN NaN
df_surv_train[feature_cols] = df_surv_train[feature_cols].fillna(0)
df_surv_test[feature_cols]  = df_surv_test[feature_cols].fillna(0)

# SCALE
scaler = StandardScaler()
df_surv_train[feature_cols] = scaler.fit_transform(df_surv_train[feature_cols])
df_surv_test[feature_cols]  = scaler.transform(df_surv_test[feature_cols])

# MODEL
model = WeibullAFTFitter(penalizer=1.0)
model.fit(df_surv_train[feature_cols + ["duration", "event"]],
          duration_col="duration",
          event_col="event")

print("Model trained with cumulative 10-min windows.")

Model trained with cumulative 10-min windows.


In [53]:
from lifelines.utils import concordance_index
import numpy as np

# ─────────────────────────────────────────────
# PREDICTION
# ─────────────────────────────────────────────

# 1️⃣ Temps médian restant prédit
df_surv_test["predicted_median"] = model.predict_median(df_surv_test[feature_cols])

# Clip pour éviter les valeurs négatives
df_surv_test["predicted_median"] = df_surv_test["predicted_median"].clip(lower=0.1)

# 2️⃣ Fonction de survie à un horizon donné (ex: 30 min)
horizon_minutes = 30
surv_at_horizon = model.predict_survival_function(df_surv_test[feature_cols], times=[horizon_minutes]).T
df_surv_test["confidence"] = 1 - surv_at_horizon.values.flatten()  # probabilité que l'événement soit arrivé

# ─────────────────────────────────────────────
# ÉVALUATION
# ─────────────────────────────────────────────

# MAE
mae = np.mean(np.abs(df_surv_test["predicted_median"] - df_surv_test["duration"]))

# C-index
c_index = concordance_index(
    df_surv_test["duration"],
    -df_surv_test["predicted_median"],  # inversé car plus petit temps = plus tôt l'événement
    df_surv_test["event"]
)

# Brier score à l'horizon
y_true = (df_surv_test["duration"] <= horizon_minutes).astype(int)
brier_score = np.mean((df_surv_test["confidence"] - y_true) ** 2)

print("\n── Évaluation ──────────────────────────────")
print(f"MAE_minutes : {mae:.2f}")
print(f"C_index     : {c_index:.4f}")
print(f"Brier_score : {brier_score:.4f}")
print(f"N_test      : {len(df_surv_test)}")


── Évaluation ──────────────────────────────
MAE_minutes : 29.18
C_index     : 0.4801
Brier_score : 0.2582
N_test      : 3935
